# Audit Sampling

Auditors rarely test every transaction or record in a population. Testing everything is time-consuming and often unnecessary — a well-chosen sample can give you the same level of confidence at a fraction of the effort.

This notebook covers:
1. Why auditors sample, and key concepts
2. How to calculate an appropriate sample size
3. Three sampling methods: random, systematic, and stratified
4. How to export your sample with metadata for documentation

We'll use a dataset of 100 vendor transactions as our population throughout.

## 1. Why Auditors Sample

**Population** — the full set of records you're auditing (e.g., all 5,000 invoices processed this year).

**Sample** — a subset of that population you actually test.

**Confidence level** — how certain you want to be that your sample reflects the population. Audits typically use 90% or 95%.

**Tolerable error rate** — the maximum rate of errors you'd accept before concluding a control has failed. Common values are 5% or 10%.

The core tradeoff: a higher confidence level or lower tolerable error rate means you need a larger sample. The relationship isn't linear — going from 90% to 95% confidence increases your sample size more than you might expect.

**When to sample vs. test everything:**
- Sample when the population is large and testing everything isn't practical
- Test everything (100%) when the population is small (e.g., only 10 journal entries), when the control only fires occasionally, or when the risk is very high

## 2. Load the Population

In [13]:
%pip install pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import math

# Load the transaction population
df = pd.read_csv('transactions.csv')

print(f"Population size: {len(df)} records")
df.head()

Population size: 100 records


,Transaction_ID,Date,Vendor,Amount,Department,Approved_By,Payment_Method
0,T0001,2024-01-03,Staples,124.50,Marketing,J. Rivera,Credit Card
1,T0002,2024-01-05,AWS,3200.00,Engineering,S. Patel,ACH
2,T0003,2024-01-07,Office Depot,87.25,HR,M. Chen,Credit Card
3,T0004,2024-01-08,Delta Airlines,1450.00,Sales,J. Rivera,Credit Card
4,T0005,2024-01-10,Adobe,599.99,Marketing,M. Chen,ACH


In [5]:
# Get a quick overview of the population before sampling
print("Transactions by Department:")
print(df['Department'].value_counts())
print()
print("Amount summary:")
print(df['Amount'].describe().round(2))

Transactions by Department:
Department
Sales          23
Engineering    20
Operations     16
Marketing      15
HR             14
IT             12
Name: count, dtype: int64

Amount summary:
count     100.00
mean     1287.50
std      1856.76
min        22.60
25%        91.64
50%       420.00
75%      1670.00
max      6500.00
Name: Amount, dtype: float64


## 3. Calculate Sample Size

A common formula for attribute sampling (testing whether something is present or absent, like an approval) is:

$$n = \frac{Z^2 \times p \times (1 - p)}{E^2}$$

Where:
- **Z** = Z-score for your confidence level (1.645 for 90%, 1.96 for 95%)
- **p** = expected error rate in the population (use 0.5 if unknown — this gives the most conservative/largest sample)
- **E** = tolerable error rate (e.g., 0.05 for 5%)

This formula assumes an infinite population. For smaller populations, apply the **finite population correction (FPC)**:

$$n_{adjusted} = \frac{n}{1 + \frac{n - 1}{N}}$$

Where **N** is the population size.

In [6]:
# --- Configure your sampling parameters here ---
CONFIDENCE_LEVEL = 0.95   # 90% = 0.90, 95% = 0.95
TOLERABLE_ERROR  = 0.05   # 5% = 0.05, 10% = 0.10
EXPECTED_ERROR   = 0.5    # Use 0.5 (most conservative) if unknown
# ------------------------------------------------

POPULATION_SIZE = len(df)

# Z-scores for common confidence levels
z_scores = {0.90: 1.645, 0.95: 1.96, 0.99: 2.576}
z = z_scores.get(CONFIDENCE_LEVEL)

if z is None:
    raise ValueError("Confidence level must be 0.90, 0.95, or 0.99")

# Base sample size (infinite population)
n_base = (z**2 * EXPECTED_ERROR * (1 - EXPECTED_ERROR)) / (TOLERABLE_ERROR**2)

# Finite population correction
n_adjusted = n_base / (1 + (n_base - 1) / POPULATION_SIZE)
SAMPLE_SIZE = math.ceil(n_adjusted)

print(f"Population size:       {POPULATION_SIZE}")
print(f"Confidence level:      {int(CONFIDENCE_LEVEL * 100)}%")
print(f"Tolerable error rate:  {int(TOLERABLE_ERROR * 100)}%")
print(f"Base sample size:      {math.ceil(n_base)}")
print(f"Adjusted sample size:  {SAMPLE_SIZE} (after finite population correction)")

Population size:       100
Confidence level:      95%
Tolerable error rate:  5%
Base sample size:      385
Adjusted sample size:  80 (after finite population correction)


Notice that the adjusted sample size is smaller than the base. When your population is small relative to the uncorrected sample size, the FPC has a meaningful impact. For very large populations, the correction is negligible.

We'll use this `SAMPLE_SIZE` across all three sampling methods below.

## 4. Method 1 — Random Sampling

Every item in the population has an equal chance of being selected. This is the simplest method and appropriate for most general audit tests where the population is relatively homogeneous.

We set a `random_state` so the sample is reproducible — running the notebook again will produce the same selection, which is important for documentation and review.

In [7]:
RANDOM_SEED = 42  # Change this to get a different random sample; document what seed you used

random_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).copy()
random_sample = random_sample.sort_values('Transaction_ID').reset_index(drop=True)

print(f"Random sample: {len(random_sample)} records")
random_sample

Random sample: 80 records


,Transaction_ID,Date,Vendor,Amount,Department,Approved_By,Payment_Method
0,T0001,2024-01-03,Staples,124.50,Marketing,J. Rivera,Credit Card
1,T0004,2024-01-08,Delta Airlines,1450.00,Sales,J. Rivera,Credit Card
2,T0005,2024-01-10,Adobe,599.99,Marketing,M. Chen,ACH
3,T0006,2024-01-11,Zoom,149.00,IT,S. Patel,ACH
4,T0007,2024-01-14,FedEx,62.10,Operations,L. Gomez,Credit Card
...,...,...,...,...,...,...,...
75,T0096,2024-05-20,Slack,320.00,IT,S. Patel,ACH
76,T0097,2024-05-21,FedEx,41.70,Operations,L. Gomez,Credit Card
77,T0098,2024-05-22,Delta Airlines,1730.00,Sales,J. Rivera,Credit Card
78,T0099,2024-05-23,Office Depot,119.50,HR,M. Chen,Credit Card


## 5. Method 2 — Systematic Sampling

Select every *k*th item from the population, starting from a random point. The interval *k* is calculated as `population size ÷ sample size`.

Systematic sampling is useful when records are already ordered (e.g., by date or transaction ID) and you want even coverage across the full period. It's also easy to explain to a reviewer — "we took every 4th transaction starting from record 2."

**Caution:** Avoid systematic sampling if the data has a pattern that aligns with your interval (e.g., if transactions are grouped in batches of 4, every 4th record could always land on the same type).

In [8]:
import random

random.seed(RANDOM_SEED)

interval = math.floor(POPULATION_SIZE / SAMPLE_SIZE)
start = random.randint(0, interval - 1)  # Random start within the first interval

systematic_indices = list(range(start, POPULATION_SIZE, interval))[:SAMPLE_SIZE]
systematic_sample = df.iloc[systematic_indices].copy().reset_index(drop=True)

print(f"Interval:          every {interval} records")
print(f"Starting index:    {start}")
print(f"Systematic sample: {len(systematic_sample)} records")
systematic_sample

Interval:          every 1 records
Starting index:    0
Systematic sample: 80 records


,Transaction_ID,Date,Vendor,Amount,Department,Approved_By,Payment_Method
0,T0001,2024-01-03,Staples,124.50,Marketing,J. Rivera,Credit Card
1,T0002,2024-01-05,AWS,3200.00,Engineering,S. Patel,ACH
2,T0003,2024-01-07,Office Depot,87.25,HR,M. Chen,Credit Card
3,T0004,2024-01-08,Delta Airlines,1450.00,Sales,J. Rivera,Credit Card
4,T0005,2024-01-10,Adobe,599.99,Marketing,M. Chen,ACH
...,...,...,...,...,...,...,...
75,T0076,2024-04-22,UPS,47.30,Operations,L. Gomez,Credit Card
76,T0077,2024-04-23,Staples,73.20,Marketing,M. Chen,Credit Card
77,T0078,2024-04-24,GitHub,420.00,Engineering,S. Patel,ACH
78,T0079,2024-04-25,Adobe,599.99,Engineering,S. Patel,ACH


## 6. Method 3 — Stratified Sampling

Divide the population into subgroups (strata) and sample from each group proportionally. Use this when the population has distinct segments that vary significantly — for example, departments with very different transaction volumes or risk levels.

Stratified sampling ensures that smaller but important subgroups aren't missed entirely, which can happen with random sampling.

Here we'll stratify by **Department**.

In [9]:
STRATIFY_COLUMN = 'Department'

# Calculate each stratum's proportion of the population
strata_counts = df[STRATIFY_COLUMN].value_counts()
strata_proportions = strata_counts / POPULATION_SIZE

print("Stratum breakdown:")
for stratum, count in strata_counts.items():
    proportion = strata_proportions[stratum]
    allocated = math.floor(SAMPLE_SIZE * proportion)
    print(f"  {stratum:<15} {count:>3} records  ({proportion:.0%})  → {allocated} samples")

Stratum breakdown:
  Sales            23 records  (23%)  → 18 samples
  Engineering      20 records  (20%)  → 16 samples
  Operations       16 records  (16%)  → 12 samples
  Marketing        15 records  (15%)  → 12 samples
  HR               14 records  (14%)  → 11 samples
  IT               12 records  (12%)  → 9 samples


In [10]:
strata_samples = []

for stratum, proportion in strata_proportions.items():
    n = math.floor(SAMPLE_SIZE * proportion)
    if n == 0:
        continue
    stratum_df = df[df[STRATIFY_COLUMN] == stratum]
    if n > len(stratum_df):
        raise ValueError(f"Stratum '{stratum}' has fewer records ({len(stratum_df)}) than required samples ({n})")
    strata_samples.append(stratum_df.sample(n=n, random_state=RANDOM_SEED))

stratified_sample = pd.concat(strata_samples)

# Fill any rounding gap by randomly selecting from the remaining records
shortfall = SAMPLE_SIZE - len(stratified_sample)
if shortfall > 0:
    already_selected = stratified_sample.index
    remaining = df.drop(index=already_selected)
    extras = remaining.sample(n=shortfall, random_state=RANDOM_SEED)
    stratified_sample = pd.concat([stratified_sample, extras])

stratified_sample = stratified_sample.sort_values('Transaction_ID').reset_index(drop=True)

print(f"Stratified sample: {len(stratified_sample)} records")
print()
print("Sample breakdown by department:")
print(stratified_sample[STRATIFY_COLUMN].value_counts())

Stratified sample: 80 records

Sample breakdown by department:
Department
Sales          18
Engineering    16
Operations     13
Marketing      12
HR             11
IT             10
Name: count, dtype: int64


## 7. Comparing the Methods

It's worth looking at how the three samples differ before deciding which to use — or which to document.

In [11]:
comparison = pd.DataFrame({
    'Method': ['Random', 'Systematic', 'Stratified'],
    'Sample Size': [len(random_sample), len(systematic_sample), len(stratified_sample)],
    'Avg Amount': [
        random_sample['Amount'].mean(),
        systematic_sample['Amount'].mean(),
        stratified_sample['Amount'].mean()
    ],
    'Unique Departments': [
        random_sample['Department'].nunique(),
        systematic_sample['Department'].nunique(),
        stratified_sample['Department'].nunique()
    ]
})

comparison['Avg Amount'] = comparison['Avg Amount'].round(2)
comparison

,Method,Sample Size,Avg Amount,Unique Departments
0,Random,80,1258.68,6
1,Systematic,80,1244.90,6
2,Stratified,80,1287.08,6


**Which method to use:**

- **Random** — default choice; easy to implement and defend. Use when the population is homogeneous.
- **Systematic** — good for ordered populations (e.g., daily transactions); provides even coverage over time.
- **Stratified** — use when subgroups differ significantly in size or risk, and you want guaranteed representation from each group.

All three methods are defensible to auditors and regulators as long as you document how the sample was selected.

## 8. Export the Sample

Documentation is as important as the sample itself. When you hand off your sample to a reviewer or include it in a workpaper, they need to know:
- What population was tested
- What method was used
- What parameters were applied
- When the sample was selected

We'll export the sample to Excel with a metadata sheet capturing all of this.

In [14]:
from datetime import datetime

# Choose which sample to export
EXPORT_METHOD = 'Stratified'  # Change to 'Random' or 'Systematic' as needed

samples_map = {
    'Random': random_sample,
    'Systematic': systematic_sample,
    'Stratified': stratified_sample
}

export_sample = samples_map[EXPORT_METHOD]

# Build a metadata summary
metadata = pd.DataFrame([
    {'Parameter': 'Population File',      'Value': 'transactions.csv'},
    {'Parameter': 'Population Size',      'Value': POPULATION_SIZE},
    {'Parameter': 'Sampling Method',      'Value': EXPORT_METHOD},
    {'Parameter': 'Confidence Level',     'Value': f"{int(CONFIDENCE_LEVEL * 100)}%"},
    {'Parameter': 'Tolerable Error Rate', 'Value': f"{int(TOLERABLE_ERROR * 100)}%"},
    {'Parameter': 'Sample Size',          'Value': len(export_sample)},
    {'Parameter': 'Random Seed',          'Value': RANDOM_SEED},
    {'Parameter': 'Date Generated',       'Value': datetime.today().strftime('%Y-%m-%d')},
])

output_file = f'Sampling_Workpaper_{EXPORT_METHOD}.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    metadata.to_excel(writer, sheet_name='Metadata', index=False)
    export_sample.to_excel(writer, sheet_name='Sample', index=False)

print(f"Exported: {output_file}")
print(f"  Sheet 'Metadata' — sampling parameters")
print(f"  Sheet 'Sample'   — {len(export_sample)} selected records")

Exported: Sampling_Workpaper_Stratified.xlsx
  Sheet 'Metadata' — sampling parameters
  Sheet 'Sample'   — 80 selected records
